# Skin Cancer Detection — Métadonnées patient + stacking (CPU, sans GPU)
Idée : un dermatologue ne regarde pas que l'image. L'**âge**, le **sexe** et la **localisation** de la lésion
aident à trancher (ex. kératose actinique → visage/sujet âgé ; nævus → sujet jeune).

On entraîne un petit modèle (« stacker ») qui combine :
- les probabilités TTA d'**EfficientNetV2-S + CBAM** et de **ConvNeXt-Tiny** (déjà calculées, aucun réentraînement) ;
- les **métadonnées** ISIC2019 (âge, sexe, localisation).

Protocole sans fuite :
- le stacker est entraîné **uniquement sur VAL** (validation croisée groupée par lésion pour choisir la config) ;
- il n'est retenu que s'il bat l'ensemble actuel **en validation croisée** ;
- le TEST n'est utilisé **qu'une fois**, à la fin.

**Inputs requis** : Outputs de `skincanerf-phase3` et `skincanerf-convnext-ensemble`, dataset ISIC2019 (cdeotte),
dataset HAM10000 metadata (kmader). Partie 1 : CPU suffit (~5 min). **Partie 2 (en bas) : GPU T4 x2 requis, ~6 h.**

## Step 1 — Setup et localisation des fichiers

In [1]:
import os, glob, json, warnings
import numpy as np
import pandas as pd
from scipy.optimize import minimize, differential_evolution
from sklearn.metrics import (accuracy_score, f1_score, recall_score, log_loss, roc_auc_score,
                             classification_report, confusion_matrix)
from sklearn.preprocessing import label_binarize, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedGroupKFold
warnings.filterwarnings("ignore")

SEED, NUM_CLASSES = 42, 7
CLASSES = ["akiec", "bcc", "bkl", "df", "nv", "mel", "vasc"]
MEL = CLASSES.index("mel")
CLASS_NAMES_FULL = ["Actinic keratoses", "Basal cell carcinoma", "Benign keratosis-like lesions",
                    "Dermatofibroma", "Melanocytic nevi", "Melanoma", "Vascular lesions"]
INPUT_ROOT, OUT_DIR = "/kaggle/input", "/kaggle/working"


def find_file(name, must_contain=None, max_depth=6):
    """Cherche un fichier par nom dans /kaggle/input (profondeur limitée)."""
    hits = []
    for d in range(1, max_depth + 1):
        hits += glob.glob(f"{INPUT_ROOT}/{'*/' * d}{name}")
    if must_contain:
        hits = [h for h in hits if must_contain(h)]
    return sorted(set(hits))


def has_cols(path, cols):
    try:
        c = set(pd.read_csv(path, nrows=0).columns)
        return all(x in c for x in cols)
    except Exception:
        return False


cx  = find_file("preds_val_convnext.npy")
eff = find_file("preds_val_final.npy")
assert cx,  "❌ Output skincanerf-convnext-ensemble introuvable → + Add Input → Notebooks"
assert eff, "❌ Output skincanerf-phase3 introuvable → + Add Input → Notebooks"
CX_DIR, EFF_DIR = os.path.dirname(cx[0]), os.path.dirname(eff[0])

isic = find_file("train.csv", lambda p: has_cols(p, ["image_name", "diagnosis"]))
assert isic, "❌ train.csv ISIC2019 introuvable → + Add Input → Datasets → jpeg-isic2019-512x512 (cdeotte)"
ISIC_CSV = isic[0]
ham = find_file("HAM10000_metadata.csv")
HAM_META = ham[0] if ham else None

print("CX_DIR   :", CX_DIR)
print("EFF_DIR  :", EFF_DIR)
print("ISIC_CSV :", ISIC_CSV)
print("HAM_META :", HAM_META or "⚠️ absent (les 197 images HAM seront traitées sans métadonnées)")

CX_DIR   : /kaggle/input/notebooks/rihembousbih/skincanerf-convnext-ensemble
EFF_DIR  : /kaggle/input/notebooks/rihembousbih/skincanerf-phase3
ISIC_CSV : /kaggle/input/datasets/cdeotte/jpeg-isic2019-512x512/train.csv
HAM_META : /kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv


## Step 2 — Splits, prédictions des 2 modèles, et contrôle de l'ensemble actuel

In [2]:
val_df  = pd.read_csv(f"{CX_DIR}/split_val.csv")
test_df = pd.read_csv(f"{CX_DIR}/split_test.csv")
y_val, y_test = val_df["label"].values, test_df["label"].values

P = {}
for s in ["val", "test"]:
    P[s] = {"eff": np.load(f"{EFF_DIR}/preds_{s}_final.npy"),
            "cx":  np.load(f"{CX_DIR}/preds_{s}_convnext.npy")}
    n = len(val_df) if s == "val" else len(test_df)
    assert len(P[s]["eff"]) == len(P[s]["cx"]) == n, f"❌ Désalignement {s}"

W = np.load(f"{CX_DIR}/ensemble_weights.npy")
ens = lambda s: W[0] * P[s]["eff"] + W[1] * P[s]["cx"]
base_val, base_test = ens("val"), ens("test")

acc = lambda yt, yp: accuracy_score(yt, yp)
mf1 = lambda yt, yp: f1_score(yt, yp, labels=np.arange(NUM_CLASSES), average="macro", zero_division=0)

a = acc(y_test, base_test.argmax(1))
print(f"Poids ensemble : EffNet={W[0]:.3f}  ConvNeXt={W[1]:.3f}")
print(f"Contrôle : TEST acc ensemble = {a:.4f} (attendu 0.8708)")
assert abs(a - 0.8708) < 0.002, "❌ Les prédictions ne correspondent pas au run ConvNeXt"

Poids ensemble : EffNet=0.488  ConvNeXt=0.512
Contrôle : TEST acc ensemble = 0.8708 (attendu 0.8708)


## Step 3 — Métadonnées (âge, sexe, localisation)
Les 197 images HAM (toutes akiec) reçoivent leurs métadonnées HAM : sinon « métadonnée manquante » deviendrait un raccourci vers akiec.

In [3]:
isic_meta = pd.read_csv(ISIC_CSV)
col_age  = next((c for c in isic_meta.columns if c.lower() in ("age_approx", "age")), None)
col_sex  = next((c for c in isic_meta.columns if c.lower() == "sex"), None)
col_site = next((c for c in isic_meta.columns if "anatom" in c.lower()), None)
print("Colonnes détectées :", {"âge": col_age, "sexe": col_sex, "site": col_site})
assert col_age and col_sex and col_site, f"❌ Colonnes manquantes. Colonnes dispo : {list(isic_meta.columns)}"

meta = isic_meta[["image_name", col_age, col_sex, col_site]].rename(
    columns={"image_name": "image_uid", col_age: "age", col_sex: "sex", col_site: "site"})

if HAM_META:
    SITE_MAP = {"face": "head/neck", "scalp": "head/neck", "ear": "head/neck", "neck": "head/neck",
                "back": "posterior torso", "chest": "anterior torso", "abdomen": "anterior torso",
                "trunk": "anterior torso", "upper extremity": "upper extremity",
                "hand": "upper extremity", "lower extremity": "lower extremity",
                "foot": "lower extremity", "acral": "palms/soles", "genital": "oral/genital"}
    h = pd.read_csv(HAM_META)
    h = pd.DataFrame({"image_uid": h["image_id"].astype(str), "age": h["age"],
                      "sex": h["sex"].replace({"unknown": np.nan}),
                      "site": h["localization"].map(SITE_MAP)})
    meta = pd.concat([meta, h[~h["image_uid"].isin(meta["image_uid"])]], ignore_index=True)

meta["sex"]  = meta["sex"].astype(str).str.lower().replace({"nan": np.nan, "none": np.nan})
meta["site"] = meta["site"].astype(str).str.lower().replace({"nan": np.nan, "none": np.nan})
meta = meta.drop_duplicates("image_uid")

val_m  = val_df[["image_uid"]].merge(meta, on="image_uid", how="left")
test_m = test_df[["image_uid"]].merge(meta, on="image_uid", how="left")
assert len(val_m) == len(val_df) and len(test_m) == len(test_df)

SEXES = sorted(meta["sex"].dropna().unique())
SITES = sorted(meta["site"].dropna().unique())
print("Sexes :", SEXES)
print("Sites :", SITES)

# Contrôle anti-raccourci : taux de métadonnées manquantes par classe (VAL)
miss = pd.DataFrame({"dx": val_df["dx"],
                     "âge manquant": val_m["age"].isna(), "sexe manquant": val_m["sex"].isna(),
                     "site manquant": val_m["site"].isna()}).groupby("dx").mean().round(3)
print("\nTaux de métadonnées manquantes par classe (VAL) :")
print(miss)

Colonnes détectées : {'âge': 'age_approx', 'sexe': 'sex', 'site': 'anatom_site_general_challenge'}
Sexes : ['female', 'male', 'unknown']
Sites : ['anterior torso', 'head/neck', 'lateral torso', 'lower extremity', 'oral/genital', 'palms/soles', 'posterior torso', 'upper extremity']

Taux de métadonnées manquantes par classe (VAL) :
       âge manquant  sexe manquant  site manquant
dx                                               
akiec         0.000            0.0          0.013
bcc           0.002            0.0          0.031
bkl           0.012            0.0          0.111
df            0.000            0.0          0.000
mel           0.019            0.0          0.044
nv            0.023            0.0          0.162
vasc          0.000            0.0          0.125


## Step 4 — Candidats et validation croisée sur VAL (groupée par lésion)

In [4]:
def logp(p):
    return np.log(np.clip(p, 1e-6, 1.0))

def X_img(s):
    return np.hstack([logp(P[s]["eff"]), logp(P[s]["cx"])])

def X_meta_onehot(m):
    age = m["age"].astype(float)
    cols = [((age.fillna(age.median() if age.notna().any() else 50) - 50) / 20).values[:, None],
            age.isna().values[:, None].astype(float)]
    cols += [(m["sex"] == s).values[:, None].astype(float) for s in SEXES]
    cols += [(m["site"] == s).values[:, None].astype(float) for s in SITES]
    cols += [m["site"].isna().values[:, None].astype(float)]
    return np.hstack(cols)

def X_meta_codes(m):
    sex  = m["sex"].map({s: i for i, s in enumerate(SEXES)}).astype(float).values
    site = m["site"].map({s: i for i, s in enumerate(SITES)}).astype(float).values
    return np.column_stack([m["age"].astype(float).values, sex, site])

FEATS = {
    "val":  {"img": X_img("val"),  "oh": X_meta_onehot(val_m),  "codes": X_meta_codes(val_m)},
    "test": {"img": X_img("test"), "oh": X_meta_onehot(test_m), "codes": X_meta_codes(test_m)},
}
N_IMG = FEATS["val"]["img"].shape[1]
cat_mask = [False] * (N_IMG + 1) + [True, True]      # [log-probs..., âge, sexe, site]

CANDIDATS = {
    "LR images seules (C=0.1)": ("img",
        lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=3000))),
    "LR images + méta (C=0.1)": ("img+oh",
        lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=3000))),
    "LR images + méta (C=1)": ("img+oh",
        lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000))),
    "GBM images + méta": ("img+codes",
        lambda: HistGradientBoostingClassifier(learning_rate=0.05, max_iter=250, max_depth=3,
                                               l2_regularization=1.0, categorical_features=cat_mask,
                                               early_stopping=False, random_state=SEED)),
}

def build_X(split, kind):
    f = FEATS[split]
    return {"img": f["img"], "img+oh": np.hstack([f["img"], f["oh"]]),
            "img+codes": np.hstack([f["img"], f["codes"]])}[kind]

def proba7(model, X):
    out = np.zeros((len(X), NUM_CLASSES))
    out[:, model.classes_] = model.predict_proba(X)
    return out

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(cv.split(np.zeros(len(y_val)), y_val, groups=val_df["lesion_id"].values))

res, OOF = {}, {}
b = base_val.argmax(1)
res["Ensemble actuel (référence)"] = (acc(y_val, b), mf1(y_val, b))
for name, (kind, make) in CANDIDATS.items():
    X = build_X("val", kind)
    oof = np.zeros((len(X), NUM_CLASSES))
    for tr, te in folds:
        m = make().fit(X[tr], y_val[tr])
        oof[te] = proba7(m, X[te])
    OOF[name] = oof
    res[name] = (acc(y_val, oof.argmax(1)), mf1(y_val, oof.argmax(1)))

print(f"{'Modèle':32s}  CV acc   CV macro-F1")
for name, (a_, f_) in res.items():
    print(f"{name:32s}  {a_:.4f}   {f_:.4f}")

Modèle                            CV acc   CV macro-F1
Ensemble actuel (référence)       0.8743   0.8295
LR images seules (C=0.1)          0.8724   0.8144
LR images + méta (C=0.1)          0.8719   0.8141
LR images + méta (C=1)            0.8700   0.8077
GBM images + méta                 0.8611   0.8070


## Step 5 — Choix (sur VAL uniquement) et entraînement final du stacker

In [5]:
ref_acc, ref_f1 = res["Ensemble actuel (référence)"]
# On maximise l'accuracy, sans sacrifier les classes rares (macro-F1 CV ≥ référence − 0.01)
eligibles = [n for n in CANDIDATS if res[n][1] >= ref_f1 - 0.01] or list(CANDIDATS)
best = max(eligibles, key=lambda n: res[n][0])
USE_STACKER = res[best][0] > ref_acc + 0.003 and res[best][1] >= ref_f1 - 0.01

if USE_STACKER:
    kind, make = CANDIDATS[best]
    stacker = make().fit(build_X("val", kind), y_val)
    probs_test = proba7(stacker, build_X("test", kind))
    probs_cal_src = OOF[best]          # probabilités hors-échantillon pour calibrer
    CHOIX = best
else:
    probs_test, probs_cal_src, CHOIX = base_test, base_val, "Ensemble actuel (stacker non retenu)"

print(f"Référence CV acc : {ref_acc:.4f}")
print(f"Meilleur candidat : {best} (CV acc {res[best][0]:.4f})")
print("→ Retenu :", CHOIX)
gain_meta = res["LR images + méta (C=0.1)"][0] - res["LR images seules (C=0.1)"][0]
print(f"\nApport des métadonnées seules (LR, CV) : {gain_meta:+.4f} d'accuracy")

Référence CV acc : 0.8743
Meilleur candidat : LR images seules (C=0.1) (CV acc 0.8724)
→ Retenu : Ensemble actuel (stacker non retenu)

Apport des métadonnées seules (LR, CV) : -0.0005 d'accuracy


## Step 6 — TEST (une seule fois) : brut, avec IC 95 % et comparaison appariée

In [6]:
s_mel  = lambda yt, yp: ((yt == MEL) & (yp == MEL)).sum() / max((yt == MEL).sum(), 1)
sp_mel = lambda yt, yp: ((yt != MEL) & (yp != MEL)).sum() / max((yt != MEL).sum(), 1)
M7 = {"Accuracy": acc, "Macro-F1": mf1, "Sensibilité mel": s_mel, "Spécificité mel": sp_mel}

def bootstrap_ci(yt, yp, metric, n=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    vals = [metric(yt[i], yp[i]) for i in (rng.integers(0, len(yt), len(yt)) for _ in range(n))]
    return np.percentile(vals, [2.5, 97.5])

def report(nom, yt, yp, metrics=M7):
    print(f"\n=== {nom} ({len(yt)} images) ===")
    out = {}
    for mname, m in metrics.items():
        v = m(yt, yp); lo, hi = bootstrap_ci(yt, yp, m)
        out[mname] = (float(v), float(lo), float(hi))
        print(f"  {mname:18s} {v:.4f}   IC95% [{lo:.4f} – {hi:.4f}]")
    return out

def paired_diff(yt, a, b, metric, n=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    d = [metric(yt[i], a[i]) - metric(yt[i], b[i])
         for i in (rng.integers(0, len(yt), len(yt)) for _ in range(n))]
    return metric(yt, a) - metric(yt, b), np.percentile(d, [2.5, 97.5])

yt_raw  = probs_test.argmax(1)
ref_raw = base_test.argmax(1)
res_test_raw = report(f"TEST brut — {CHOIX}", y_test, yt_raw)
yb = label_binarize(y_test, classes=np.arange(NUM_CLASSES))
print(f"  Macro AUC          {roc_auc_score(yb, probs_test, average='macro'):.4f}")

gains = {}
print("\nGain vs ensemble actuel (bootstrap apparié) :")
for mname, m in [("Accuracy", acc), ("Macro-F1", mf1)]:
    d, (lo, hi) = paired_diff(y_test, yt_raw, ref_raw, m)
    gains[mname] = (float(d), float(lo), float(hi))
    verdict = "✅ significatif" if lo > 0 else ("❌ moins bon" if hi < 0 else "≈ non significatif")
    print(f"  {mname:9s} {d:+.4f}   IC95% [{lo:+.4f} ; {hi:+.4f}]   {verdict}")

print("\n--- Rapport par classe (brut) ---")
print(classification_report(y_test, yt_raw, labels=np.arange(NUM_CLASSES),
                            target_names=CLASS_NAMES_FULL, digits=4, zero_division=0))


=== TEST brut — Ensemble actuel (stacker non retenu) (3738 images) ===
  Accuracy           0.8708   IC95% [0.8606 – 0.8810]
  Macro-F1           0.8309   IC95% [0.8048 – 0.8528]
  Sensibilité mel    0.7304   IC95% [0.6964 – 0.7638]
  Spécificité mel    0.9662   IC95% [0.9597 – 0.9724]
  Macro AUC          0.9807

Gain vs ensemble actuel (bootstrap apparié) :
  Accuracy  +0.0000   IC95% [+0.0000 ; +0.0000]   ≈ non significatif
  Macro-F1  +0.0000   IC95% [+0.0000 ; +0.0000]   ≈ non significatif

--- Rapport par classe (brut) ---
                               precision    recall  f1-score   support

            Actinic keratoses     0.7655    0.7255    0.7450       153
         Basal cell carcinoma     0.8793    0.9338    0.9057       468
Benign keratosis-like lesions     0.7719    0.7519    0.7618       387
               Dermatofibroma     0.8462    0.7857    0.8148        28
             Melanocytic nevi     0.9063    0.9403    0.9230      1976
                     Melanoma     0.8

## Step 7 — Version calibrée (sensibilité mélanome ≥ 0.85), calibrée sur les probabilités hors-échantillon de VAL

In [7]:
CIBLE_SENS_MEL = 0.85

def apply_temperature(probs, T):
    s = np.exp(np.log(np.clip(probs, 1e-12, 1 - 1e-12)) / T)
    return s / s.sum(axis=1, keepdims=True)

def find_temperature(pv, y):
    nll = lambda x: log_loss(y, apply_temperature(pv, float(x[0])), labels=np.arange(NUM_CLASSES))
    return float(minimize(nll, x0=[1.0], bounds=[(0.05, 5.0)], method="L-BFGS-B").x[0])

def apply_thresholds(probs, thr):
    return np.argmax(probs / np.clip(np.asarray(thr, float), 0.05, 5.0)[None, :], axis=1)

def optimize_thresholds(pv, y):
    def obj(thr):
        pred = apply_thresholds(pv, thr)
        sens = recall_score(y, pred, labels=[MEL], average="macro", zero_division=0)
        return 10.0 + (CIBLE_SENS_MEL - sens) * 100.0 if sens < CIBLE_SENS_MEL else -accuracy_score(y, pred)
    r = differential_evolution(obj, [(0.2, 3.0)] * NUM_CLASSES, maxiter=80, popsize=10,
                               polish=False, seed=SEED)
    return np.clip(r.x, 0.2, 3.0)

T   = find_temperature(probs_cal_src, y_val)
thr = optimize_thresholds(apply_temperature(probs_cal_src, T), y_val)
yt_cal = apply_thresholds(apply_temperature(probs_test, T), thr)
print(f"T = {T:.3f} | seuils :", "  ".join(f"{c}={t:.2f}" for c, t in zip(CLASSES, thr)))
res_test_cal = report(f"TEST calibré — {CHOIX}", y_test, yt_cal)
print("\nMatrice de confusion (calibré ; lignes = vrai) :")
print(confusion_matrix(y_test, yt_cal, labels=np.arange(NUM_CLASSES)))

T = 0.678 | seuils : akiec=1.84  bcc=2.93  bkl=2.66  df=2.09  nv=2.58  mel=0.44  vasc=0.50

=== TEST calibré — Ensemble actuel (stacker non retenu) (3738 images) ===
  Accuracy           0.8414   IC95% [0.8299 – 0.8526]
  Macro-F1           0.8104   IC95% [0.7828 – 0.8332]
  Sensibilité mel    0.8522   IC95% [0.8247 – 0.8783]
  Spécificité mel    0.8924   IC95% [0.8816 – 0.9032]

Matrice de confusion (calibré ; lignes = vrai) :
[[ 110    8   13    0    2   20    0]
 [  15  419    3    1    8   21    1]
 [  22   13  250    0   27   75    0]
 [   0    1    0   22    1    4    0]
 [   0   14   26    1 1724  205    6]
 [   5    7    9    0   80  588    1]
 [   0    0    0    0    1    3   32]]


## Step 8 — Sauvegarde
Note : PH2 n'a pas de métadonnées → pour PH2, le modèle de référence reste l'ensemble images seules.

In [8]:
np.save(f"{OUT_DIR}/probs_test_stacker.npy", probs_test)
np.save(f"{OUT_DIR}/temperature_stacker.npy", np.array([T]))
np.save(f"{OUT_DIR}/thresholds_stacker.npy", thr)
if USE_STACKER:
    import pickle
    with open(f"{OUT_DIR}/stacker.pkl", "wb") as f:
        pickle.dump({"model": stacker, "kind": CANDIDATS[best][0],
                     "sexes": SEXES, "sites": SITES}, f)

config = {"choix": CHOIX, "cv": {k: [float(v[0]), float(v[1])] for k, v in res.items()},
          "apport_meta_cv_acc": float(gain_meta), "gain_vs_ensemble_test": gains,
          "test_brut": res_test_raw, "test_calibre": res_test_cal,
          "temperature": T, "seuils": thr.tolist()}
with open(f"{OUT_DIR}/run_config_stacker.json", "w") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print(json.dumps({k: config[k] for k in ["choix", "gain_vs_ensemble_test"]}, indent=2, ensure_ascii=False))

{
  "choix": "Ensemble actuel (stacker non retenu)",
  "gain_vs_ensemble_test": {
    "Accuracy": [
      0.0,
      0.0,
      0.0
    ],
    "Macro-F1": [
      0.0,
      0.0,
      0.0
    ]
  }
}


---
# PARTIE 2 — GPU : anti-NaN, fin d'entraînement ConvNeXt, 3ᵉ modèle (EfficientNet-B4), ensemble final
Objectif : dépasser **88 %** d'accuracy TEST sans fuite.

| Étape | Rôle | Durée (T4 x2) |
|---|---|---|
| 2.1 – 2.2 | Setup GPU + pipeline (mixup corrigé, filtre anti-NaN) | 1 min |
| 2.3 | Diagnostic : le pipeline produit-il des valeurs non finies ? | ~5 min |
| 2.4 | `NanGuard` : restaure le dernier état sain au lieu d'arrêter | – |
| 2.5 | ConvNeXt-Tiny : reprise de l'entraînement interrompu | ~1 h 45 |
| 2.6 | EfficientNet-B4 : 3ᵉ modèle | ~3 h |
| 2.7 | Prédictions TTA des nouveaux modèles | ~20 min |
| 2.8 – 2.9 | Choix de l'ensemble **sur VAL** + calibration (1 seul facteur mélanome) | 1 min |
| 2.10 – 2.12 | TEST (une seule fois), PH2, sauvegarde | 2 min |

**Avant de lancer :**
- Settings → **Accelerator : GPU T4 x2**, **Internet : On** (poids ImageNet de B4).
- Inputs : en plus de ceux de la partie 1, les **images** HAM10000 (surajghuwalewala) et PH2 (spacesurfer).
- Un garde-fou arrête proprement les entraînements à 9 h 30 : les checkpoints déjà écrits restent utilisables.
- Si la session coupe, relancez **Run All** : les prédictions TTA déjà calculées sont relues depuis `/kaggle/working` si vous ajoutez l'output de ce notebook en Input (sinon elles sont recalculées).

## 2.1 — Setup GPU, constantes, splits

In [9]:
import os, time, shutil, gc, random
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, mixed_precision
from tensorflow.keras.optimizers.schedules import CosineDecay

T0       = time.time()
BUDGET_H = 9.5            # arrêt propre des entraînements après 9 h 30 (Kaggle coupe à 12 h)

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
gpus = tf.config.list_physical_devices("GPU")
assert gpus, "❌ Pas de GPU → Settings → Accelerator → GPU T4 x2"
for g in gpus:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass
mixed_precision.set_global_policy("mixed_float16")
strategy = tf.distribute.MirroredStrategy(cross_device_ops=tf.distribute.ReductionToOneDevice())
print("TF", tf.__version__, "| GPUs :", len(gpus), "| répliques :", strategy.num_replicas_in_sync)

IMG_SIZE, BATCH_SIZE = 384, 32
EVAL_RESIZE = int(IMG_SIZE * 1.15)
AUTOTUNE    = tf.data.AUTOTUNE
LABEL_SMOOTH, WEIGHT_DECAY, MIXUP_ALPHA = 0.1, 1e-4, 0.1
PREPROCESS  = lambda x: x          # ConvNeXt et EfficientNet : preprocessing intégré (entrée 0-255)
IDX_TO_CLASS = dict(enumerate(CLASSES))

train_df = pd.read_csv(f"{CX_DIR}/split_train.csv")
val_df   = pd.read_csv(f"{CX_DIR}/split_val.csv")
test_df  = pd.read_csv(f"{CX_DIR}/split_test.csv")
ph2_df   = pd.read_csv(f"{CX_DIR}/split_ph2.csv")
y_train, y_val, y_test, y_ph2 = (d["label"].values for d in (train_df, val_df, test_df, ph2_df))

for nom, d in [("train", train_df), ("val", val_df), ("test", test_df), ("ph2", ph2_df)]:
    manquants = (~d["path"].map(os.path.exists)).sum()
    print(f"{nom:5s}: {len(d):6d} images, {manquants} introuvables")
    assert manquants == 0, (f"❌ Images {nom} introuvables (ex : {d['path'].iloc[0]}) → + Add Input : "
                            "ISIC2019 (cdeotte), HAM10000 (surajghuwalewala), PH2 (spacesurfer)")
for a, b, nom in [(train_df, val_df, "train/val"), (train_df, test_df, "train/test"), (val_df, test_df, "val/test")]:
    assert not set(a["lesion_id"]) & set(b["lesion_id"]), f"❌ lésions communes {nom}"
print("✅ Splits d'origine relus, 0 lésion commune.")

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


I0000 00:00:1790207739.071291      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790207739.074541      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


TF 2.20.0 | GPUs : 2 | répliques : 2
train:  17455 images, 0 introuvables
val  :   3707 images, 0 introuvables
test :   3738 images, 0 introuvables
ph2  :    197 images, 0 introuvables
✅ Splits d'origine relus, 0 lésion commune.


## 2.2 — Pipeline tf.data (identique aux runs précédents) + 2 corrections anti-NaN
- **mixup** : `lam = g1 / (g1 + g2 + 1e-7)` (évite 0/0 quand les deux tirages Gamma valent 0) ;
- **filtre** : tout exemple ou lot contenant une valeur non finie est écarté (`safe=True`).

In [10]:
def decode_image(path):
    img_bytes = tf.io.read_file(path)
    ext = tf.strings.lower(tf.strings.split(path, ".")[-1])
    def _jpg(): return tf.image.decode_jpeg(img_bytes, channels=3)
    def _png(): return tf.image.decode_png(img_bytes, channels=3)
    def _bmp(): return tf.image.decode_bmp(img_bytes, channels=3)
    img = tf.case([(tf.equal(ext, "jpg"), _jpg), (tf.equal(ext, "jpeg"), _jpg),
                   (tf.equal(ext, "png"), _png), (tf.equal(ext, "bmp"), _bmp)],
                  default=_jpg, exclusive=True)
    return tf.ensure_shape(img, [None, None, 3])


def shades_of_gray(img, p=6.0):
    flat  = tf.reshape(img, [-1, 3])
    illum = tf.pow(tf.reduce_mean(tf.pow(flat + 1e-6, p), axis=0), 1.0 / p)
    illum = illum / (tf.norm(illum) + 1e-6)
    img   = img / (illum * tf.sqrt(3.0) + 1e-6)
    return tf.clip_by_value(img, 0.0, 255.0)


def augment_train(img):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
    img = img / 255.0
    img = tf.image.random_brightness(img, 0.30)
    img = tf.image.random_contrast(img, 0.6, 1.4)
    img = tf.image.random_saturation(img, 0.6, 1.4)
    img = tf.image.random_hue(img, 0.08)
    img = tf.clip_by_value(img, 0.0, 1.0) * 255.0
    h  = tf.random.uniform([], IMG_SIZE // 8, IMG_SIZE // 4, dtype=tf.int32)
    y0 = tf.random.uniform([], 0, IMG_SIZE - h, dtype=tf.int32)
    x0 = tf.random.uniform([], 0, IMG_SIZE - h, dtype=tf.int32)
    yy, xx = tf.range(IMG_SIZE)[:, None], tf.range(IMG_SIZE)[None, :]
    inside = (yy >= y0) & (yy < y0 + h) & (xx >= x0) & (xx < x0 + h)
    apply_cut = tf.cast(tf.random.uniform([]) < 0.5, tf.float32)
    img = img * (1.0 - apply_cut * tf.cast(inside, tf.float32)[:, :, None])
    return tf.ensure_shape(img, [IMG_SIZE, IMG_SIZE, 3])


def load_and_preprocess(path, label, training=False):
    img = tf.cast(decode_image(path), tf.float32)
    if training:
        shape = tf.shape(img)
        scale = tf.random.uniform([], 0.7, 1.0)
        h = tf.cast(tf.cast(shape[0], tf.float32) * scale, tf.int32)
        w = tf.cast(tf.cast(shape[1], tf.float32) * scale, tf.int32)
        img = tf.image.random_crop(img, [h, w, 3])
        img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
        img = augment_train(img)
    else:
        img = tf.image.resize(img, (EVAL_RESIZE, EVAL_RESIZE))
        off = (EVAL_RESIZE - IMG_SIZE) // 2
        img = tf.image.crop_to_bounding_box(img, off, off, IMG_SIZE, IMG_SIZE)
    img = PREPROCESS(shades_of_gray(img))
    return img, tf.one_hot(label, NUM_CLASSES)


def _finite(x, y):
    return tf.logical_and(tf.reduce_all(tf.math.is_finite(x)), tf.reduce_all(tf.math.is_finite(y)))


def make_dataset(df, training=False, shuffle_buffer=4096, batched=True, safe=True):
    ds = tf.data.Dataset.from_tensor_slices((df["path"].values.astype(str),
                                             df["label"].values.astype(np.int32)))
    if training:
        ds = ds.shuffle(shuffle_buffer, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: load_and_preprocess(p, y, training=training), num_parallel_calls=AUTOTUNE)
    if safe and training:
        ds = ds.filter(_finite)
    if batched:
        ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


def mixup(ds, alpha=MIXUP_ALPHA, safe=True):
    def _mix(imgs, labels):
        b   = tf.shape(imgs)[0]
        g1  = tf.random.gamma([b], alpha)
        g2  = tf.random.gamma([b], alpha)
        lam = tf.clip_by_value(g1 / (g1 + g2 + 1e-7), 0.0, 1.0)      # CORRIGÉ : plus de 0/0
        idx = tf.random.shuffle(tf.range(b))
        li, ll = tf.reshape(lam, [b, 1, 1, 1]), tf.reshape(lam, [b, 1])
        return (li * imgs + (1 - li) * tf.gather(imgs, idx),
                ll * labels + (1 - ll) * tf.gather(labels, idx))
    ds = ds.map(_mix, num_parallel_calls=AUTOTUNE)
    return ds.filter(_finite) if safe else ds


def balanced_dataset(df, power=0.5, safe=True):
    dss, weights = [], []
    for c in CLASSES:
        sub = df[df["dx"] == c]
        if len(sub) == 0:
            continue
        dss.append(make_dataset(sub, training=True, shuffle_buffer=min(len(sub), 4096),
                                batched=False, safe=safe).repeat())
        weights.append(float(len(sub)) ** power)
    w = np.array(weights) / np.sum(weights)
    ds = tf.data.Dataset.sample_from_datasets(dss, weights=list(w), seed=SEED)
    opts = tf.data.Options()
    opts.experimental_distribute.auto_shard_policy = tf.data.experimental.AutoShardPolicy.DATA
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE).with_options(opts)


val_ds = make_dataset(val_df, training=False)
print("Pipeline défini (mixup corrigé + filtre anti-NaN).")

Pipeline défini (mixup corrigé + filtre anti-NaN).


## 2.3 — Diagnostic : le pipeline produit-il des valeurs non finies ?
Parcourt `DIAG_BATCHES` lots du pipeline d'entraînement **sans filtre et sans modèle**.
- s'il trouve des lots non finis → la cause des NaN était les données (le filtre de 2.2 la neutralise) ;
- s'il n'en trouve aucun → la cause est côté modèle / float16 : c'est `NanGuard` (2.4) qui protège l'entraînement.

Mettre `RUN_DIAG = False` pour sauter cette étape.

In [11]:
RUN_DIAG, DIAG_BATCHES = True, 1500

if RUN_DIAG:
    t0, n_bad, vmin, vmax = time.time(), 0, np.inf, -np.inf
    ds_raw = mixup(balanced_dataset(train_df, safe=False), safe=False)
    for i, (x, y) in enumerate(ds_raw.take(DIAG_BATCHES)):
        fx = bool(tf.reduce_all(tf.math.is_finite(x)))
        fy = bool(tf.reduce_all(tf.math.is_finite(y)))
        if fx:
            vmin, vmax = min(vmin, float(tf.reduce_min(x))), max(vmax, float(tf.reduce_max(x)))
        if not (fx and fy):
            n_bad += 1
            print(f"  ⚠️ lot {i} : images finies={fx}  labels finis={fy}")
    print(f"\n{DIAG_BATCHES} lots contrôlés en {time.time() - t0:.0f}s — lots non finis : {n_bad}")
    print(f"Plage des pixels : [{vmin:.1f} ; {vmax:.1f}] (attendu dans [0 ; 255])")
    print("→ Cause probable des NaN :", "LES DONNÉES (neutralisées par le filtre)" if n_bad
          else "le modèle / float16 (pas les données) → NanGuard prend le relais")
    del ds_raw


1500 lots contrôlés en 511s — lots non finis : 0
Plage des pixels : [0.0 ; 255.0] (attendu dans [0 ; 255])
→ Cause probable des NaN : le modèle / float16 (pas les données) → NanGuard prend le relais


## 2.4 — `NanGuard` et entraînement robuste
Au lieu d'**arrêter** l'entraînement au premier NaN (TerminateOnNaN), `NanGuard` :
1. photographie les poids **et** l'état de l'optimiseur tous les 50 lots (seulement s'ils sont sains) ;
2. si la loss ou les poids deviennent non finis, restaure cette photo et **continue** — le lot fautif est dépassé.

On perd au plus 50 lots par incident, au lieu d'une époque entière + un redémarrage de l'optimiseur.

NB : après un incident, la loss affichée en fin d'époque ne couvre que les lots suivant la restauration (affichage seulement).

In [12]:
class MacroF1(keras.callbacks.Callback):
    def __init__(self, ds, y):
        super().__init__(); self.ds, self.y = ds, y
    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        p = self.model.predict(self.ds, verbose=0)
        logs["val_macro_f1"] = f1_score(self.y, p.argmax(1), average="macro") if np.isfinite(p).all() else 0.0
        print(f"   époque {epoch + 1} : val_macro_f1 = {logs['val_macro_f1']:.4f}  "
              f"({(time.time() - T0) / 3600:.2f} h écoulées)")


class NanGuard(keras.callbacks.Callback):
    def __init__(self, every=50, max_rollbacks=30):
        super().__init__()
        self.every, self.max_rb, self.n_rb = every, max_rollbacks, 0
        self.w = self.o = None

    def _opt_vars(self):
        opt, out, seen = self.model.optimizer, [], set()
        for o in (opt, getattr(opt, "inner_optimizer", None)):
            for v in (o.variables if o is not None else []):
                if id(v) not in seen:
                    seen.add(id(v)); out.append(v)
        return out

    def _healthy(self):
        return all(np.isfinite(v.numpy()).all() for v in self.model.trainable_weights[-2:])

    def _snapshot(self):
        self.w = [v.numpy() for v in self.model.weights]
        self.o = [v.numpy() for v in self._opt_vars()]

    def _restore(self):
        for v, a in zip(self.model.weights, self.w):
            v.assign(a)
        ov = self._opt_vars()
        if self.o is not None and len(ov) == len(self.o):
            for v, a in zip(ov, self.o):
                v.assign(a)
        for m in self.model.metrics:
            m.reset_state()

    def on_train_begin(self, logs=None):
        self._snapshot()

    def on_train_batch_end(self, batch, logs=None):
        loss = (logs or {}).get("loss")
        bad = (loss is not None and not np.isfinite(float(loss))) or not self._healthy()
        if bad:
            self.n_rb += 1
            print(f"\n   🛟 NaN au lot {batch} → restauration du dernier état sain "
                  f"(incident {self.n_rb}/{self.max_rb})")
            if self.n_rb > self.max_rb:
                print("   ❌ Trop d'incidents : arrêt (le meilleur checkpoint est conservé).")
                self._restore(); self.model.stop_training = True
            else:
                self._restore()
            return
        if batch % self.every == 0:
            self._snapshot()


class TimeBudget(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if (time.time() - T0) / 3600 > BUDGET_H:
            print("\n⏱️ Budget temps atteint → arrêt propre (meilleur checkpoint conservé).")
            self.model.stop_training = True


def make_optimizer(peak_lr, n_epochs, warmup_epochs=1):
    spe = max(len(train_df) // BATCH_SIZE, 1)
    sched = CosineDecay(initial_learning_rate=peak_lr / 10, decay_steps=spe * n_epochs,
                        warmup_target=peak_lr, warmup_steps=spe * warmup_epochs, alpha=0.01)
    return keras.optimizers.AdamW(learning_rate=sched, weight_decay=WEIGHT_DECAY, clipnorm=1.0)


CE_SMOOTH = keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH)


def fit_safe(model, train_ds, steps, epochs, peak_lr, ckpt, best=None, patience=6, warmup=1):
    # Entraîne avec NanGuard ; le checkpoint n'est écrasé que si val_macro_f1 bat `best`.
    if (time.time() - T0) / 3600 > BUDGET_H:
        print("⏱️ Budget temps déjà atteint : entraînement sauté.")
        return best, 0
    with strategy.scope():
        model.compile(optimizer=make_optimizer(peak_lr, epochs, warmup),
                      loss=CE_SMOOTH, metrics=["accuracy"])
    print("Optimiseur :", type(model.optimizer).__name__,
          "| inner :", type(getattr(model.optimizer, "inner_optimizer", None)).__name__)
    guard = NanGuard()
    cbs = [MacroF1(val_ds, y_val), guard,
           keras.callbacks.ModelCheckpoint(ckpt, monitor="val_macro_f1", mode="max",
                                           save_best_only=True, initial_value_threshold=best),
           keras.callbacks.EarlyStopping(monitor="val_macro_f1", mode="max", patience=patience,
                                         restore_best_weights=True),
           TimeBudget()]
    h = model.fit(train_ds, steps_per_epoch=steps, validation_data=val_ds,
                  epochs=epochs, callbacks=cbs, verbose=2)
    scores = [v for v in h.history.get("val_macro_f1", []) if np.isfinite(v)]
    best_run = max(scores) if scores else -np.inf
    print(f"→ {len(scores)} époques | meilleur val_macro_f1 = {best_run:.4f} | incidents NaN : {guard.n_rb}")
    return max(best if best is not None else -np.inf, best_run), len(scores)


print("NanGuard, callbacks et fit_safe définis.")

NanGuard, callbacks et fit_safe définis.


## 2.5 — ConvNeXt-Tiny : reprise de l'entraînement interrompu par les NaN
Le run précédent s'est arrêté à 18 époques utiles alors que le val macro-F1 montait encore.
On repart de son meilleur checkpoint avec un lr plus bas ; le nouveau checkpoint n'est écrit que s'il fait mieux.

In [13]:
CX_OLD, CX_NEW = f"{CX_DIR}/convnext_tiny_final.keras", f"{OUT_DIR}/convnext_tiny_final_v2.keras"
EPOCHS_CX, LR_CX = 12, 2e-5

with strategy.scope():
    model = keras.models.load_model(CX_OLD, compile=False)
p = model.predict(val_ds, verbose=0)
f1_cx_old = f1_score(y_val, p.argmax(1), average="macro")
print(f"ConvNeXt de départ : val_macro_f1 (sans TTA) = {f1_cx_old:.4f}")
shutil.copy(CX_OLD, CX_NEW)                  # point de départ ; écrasé seulement si amélioration

for l in model.layers:                       # ConvNeXt : LayerNorm, pas de BatchNorm
    l.trainable = True
train_ds_bal = mixup(balanced_dataset(train_df, power=0.5))
STEPS = len(train_df) // BATCH_SIZE

best_cx2, ep_cx2 = fit_safe(model, train_ds_bal, STEPS, EPOCHS_CX, LR_CX, CX_NEW,
                            best=f1_cx_old, patience=5, warmup=1)
print(f"ConvNeXt v2 : {f1_cx_old:.4f} → {best_cx2:.4f}")
del model; gc.collect()

I0000 00:00:1790208281.805030      97 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


ConvNeXt de départ : val_macro_f1 (sans TTA) = 0.7877
Optimiseur : LossScaleOptimizer | inner : AdamW
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/

2026-09-24 00:08:38.095845: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-24 00:08:38.225481: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-24 00:08:38.236018: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-24 00:08:38.385633: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-24 00:08:38.527329: E external/local_xla/xla/stream_

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
   époque 1 : val_macro_f1 = 0.7805  (0.42 h écoulées)
545/545 - 846s - 2s/step - accuracy: 0.8754 - loss: 0.8137 - val_accuracy: 0.8438 - val_loss: 0.7860 - val_macro_f1: 0.7805
Epoch 2/12
   époque 2 : val_macro_f1 = 0.7868  (0.57 h écoulées)
545/545 - 535s - 982ms/step - accuracy: 0.8736 - loss: 0.8158 - val_accuracy: 0.8379 - val_loss: 0.8024 - val_macro_f1: 0.7868
Epoch 3/12
   époque 3 : val_macro_f1 = 0.7803  (0.72 h écoulées)
545/545 - 532s - 977ms/step - accuracy: 0.8877 - loss: 0.7951 - val_accuracy: 0.8435 - val_loss: 0.7824 - val_macro_f1: 0.7803
Epoch 4/12
   époque 4 : val_macro_f1 = 0.7861  (0.86 h écoulées)
545/545 - 532s - 976ms/step - accuracy: 0.8917 - loss: 0.7784 - val_accuracy: 0.8481 - va

190218

## 2.6 — 3ᵉ modèle : EfficientNet-B4 (famille éprouvée sur ISIC, architecture différente de V2-S + CBAM)
Phase 1 : tête seule (backbone gelé). Phase 2 : fine-tuning complet (BatchNorm gelées), mixup + échantillonnage équilibré.

In [14]:
from tensorflow.keras.applications import EfficientNetB4

B4_P1, B4_FINAL = f"{OUT_DIR}/effb4_phase1.keras", f"{OUT_DIR}/effb4_final.keras"
EPOCHS_B4_P1, EPOCHS_B4_P2, LR_B4 = 4, 22, 4e-5


def build_b4():
    base = EfficientNetB4(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
    x   = layers.GlobalAveragePooling2D(name="gap", dtype="float32")(base.output)
    x   = layers.Dropout(0.4, name="head_drop", dtype="float32")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", name="pred", dtype="float32")(x)
    return Model(base.input, out, name="effb4_skin"), base


with strategy.scope():
    model, base = build_b4()
print(f"EfficientNet-B4 : {model.count_params() / 1e6:.1f} M paramètres")

# Phase 1 : tête seule
base.trainable = False
best_b4_p1, _ = fit_safe(model, make_dataset(train_df, training=True).repeat(), STEPS, EPOCHS_B4_P1,
                         1e-3, B4_P1, patience=3, warmup=1)

# Phase 2 : fine-tuning complet, BatchNorm gelées
if os.path.exists(B4_P1):
    model.load_weights(B4_P1)
    shutil.copy(B4_P1, B4_FINAL)
    for l in model.layers:
        l.trainable = not isinstance(l, layers.BatchNormalization)
    best_b4, ep_b4 = fit_safe(model, train_ds_bal, STEPS, EPOCHS_B4_P2, LR_B4, B4_FINAL,
                              best=best_b4_p1, patience=6, warmup=2)
    print(f"EfficientNet-B4 : phase 1 {best_b4_p1:.4f} → phase 2 {best_b4:.4f}")
else:
    print("⚠️ Pas de checkpoint de phase 1 : B4 sera ignoré.")
del model, base; gc.collect()

71686520/71686520 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
EfficientNet-B4 : 17.7 M paramètres
Optimiseur : LossScaleOptimizer | inner : AdamW
Epoch 1/4
   époque 1 : val_macro_f1 = 0.3198  (2.15 h écoulées)
545/545 - 362s - 664ms/step - accuracy: 0.5832 - loss: 1.3386 - val_accuracy: 0.6501 - val_loss: 1.2132 - val_macro_f1: 0.3198
Epoch 2/4
   époque 2 : val_macro_f1 = 0.3715  (2.23 h écoulées)
545/545 - 270s - 495ms/step - accuracy: 0.6281 - loss: 1.2238 - val_accuracy: 0.6639 - val_loss: 1.1695 - val_macro_f1: 0.3715
Epoch 3/4
   époque 3 : val_macro_f1 = 0.3765  (2.30 h écoulées)
545/545 - 255s - 468ms/step - accuracy: 0.6357 - loss: 1.1999 - val_accuracy: 0.6690 - val_loss: 1.1510 - val_macro_f1: 0.3765
Epoch 4/4
   époque 4 : val_macro_f1 = 0.3801  (2.37 h écoulées)
545/545 - 253s - 465ms/step - accuracy: 0.6402 - loss: 1.1910 - val_accuracy: 0.6771 - val_loss: 1.1395 - val_macro_f1: 0.3801
→ 4 époques | meilleur val_macro_f1 = 0.3801 | incidents NaN : 0
Optimiseur : LossScaleOptimizer 

123942

## 2.7 — Prédictions TTA (4 rotations × 2 flips) des nouveaux modèles sur VAL / TEST / PH2

In [15]:
def predict_tta(model, df, n_rot=4):
    probs = np.zeros((len(df), NUM_CLASSES), dtype=np.float64)
    for k in range(n_rot):
        for flip in (False, True):
            def _map(p, y, k=k, flip=flip):
                img = tf.cast(decode_image(p), tf.float32)
                img = tf.image.resize(img, (EVAL_RESIZE, EVAL_RESIZE))
                off = (EVAL_RESIZE - IMG_SIZE) // 2
                img = tf.image.crop_to_bounding_box(img, off, off, IMG_SIZE, IMG_SIZE)
                img = tf.image.rot90(img, k=k)
                if flip:
                    img = tf.image.flip_left_right(img)
                return PREPROCESS(shades_of_gray(img)), y
            ds = (tf.data.Dataset.from_tensor_slices((df["path"].values.astype(str),
                                                      df["label"].values.astype(np.int32)))
                  .map(_map, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE))
            probs += model.predict(ds, verbose=0)
    return probs / (n_rot * 2)


SPLITS = {"val": val_df, "test": test_df, "ph2": ph2_df}

# Modèles existants (réutilisés, pas de recalcul)
P.setdefault("ph2", {})
P["ph2"]["eff"] = np.load(f"{EFF_DIR}/preds_ph2_final.npy")
P["ph2"]["cx"]  = np.load(f"{CX_DIR}/preds_ph2_convnext.npy")

# Nouveaux modèles — nom du cache lié au fichier du modèle (évite de relire un cache périmé)
NEW_MODELS = {"cx2": CX_NEW, "b4": B4_FINAL}
for name, path in NEW_MODELS.items():
    if not os.path.exists(path):
        print(f"⏭️ {name} : checkpoint absent, ignoré."); continue
    tag = f"{name}_{int(os.path.getmtime(path))}"
    cached = {s: f"{OUT_DIR}/preds_{s}_{tag}.npy" for s in SPLITS}
    if all(os.path.exists(f) for f in cached.values()):
        for s in SPLITS:
            P[s][name] = np.load(cached[s])
        print(f"{name} : prédictions relues depuis le cache"); continue
    with strategy.scope():
        m = keras.models.load_model(path, compile=False)
    for s, d in SPLITS.items():
        t0 = time.time()
        P[s][name] = predict_tta(m, d)
        np.save(cached[s], P[s][name])
        print(f"  {s:4s}/{name:4s} : {time.time() - t0:.0f}s")
    del m; gc.collect()

MODEL_LABELS = {"eff": "EfficientNetV2-S + CBAM", "cx": "ConvNeXt-Tiny (v1)",
                "cx2": "ConvNeXt-Tiny (v2)", "b4": "EfficientNet-B4"}
AVAILABLE = [n for n in ["eff", "cx", "cx2", "b4"] if n in P["val"]]
print(f"\n{'Modèle':26s} VAL acc  VAL F1  | TEST acc  TEST F1")
for n in AVAILABLE:
    pv, pt = P["val"][n].argmax(1), P["test"][n].argmax(1)
    print(f"{MODEL_LABELS[n]:26s} {acc(y_val, pv):.4f}  {mf1(y_val, pv):.4f} | "
          f"{acc(y_test, pt):.4f}   {mf1(y_test, pt):.4f}")

  val /cx2  : 294s
  test/cx2  : 263s
  ph2 /cx2  : 39s
  val /b4   : 268s
  test/b4   : 279s
  ph2 /b4   : 33s

Modèle                     VAL acc  VAL F1  | TEST acc  TEST F1
EfficientNetV2-S + CBAM    0.8632  0.8010 | 0.8604   0.7895
ConvNeXt-Tiny (v1)         0.8530  0.7964 | 0.8537   0.7971
ConvNeXt-Tiny (v2)         0.8630  0.8098 | 0.8612   0.8269
EfficientNet-B4            0.8651  0.8277 | 0.8558   0.8022


## 2.8 — Choix de l'ensemble **sur VAL uniquement**
Candidats : toutes les combinaisons d'au moins 2 modèles, en **moyenne simple** (aucun poids optimisé → pas de sur-apprentissage de VAL).
Critère : accuracy VAL, avec garde-fou macro-F1 (≥ référence − 0.005). Un candidat ne remplace la référence
(ensemble EffNet + ConvNeXt v1 actuel) que s'il la bat d'au moins 0.3 point d'accuracy VAL.

In [16]:
from itertools import combinations

def avg(split, names):
    return np.mean([P[split][n] for n in names], axis=0)

REF_NAMES = ["eff", "cx"]
ref_val, ref_test, ref_ph2 = (W[0] * P[s]["eff"] + W[1] * P[s]["cx"] for s in ("val", "test", "ph2"))
ref_acc, ref_f1 = acc(y_val, ref_val.argmax(1)), mf1(y_val, ref_val.argmax(1))

cands = []
for r in range(2, len(AVAILABLE) + 1):
    for combo in combinations(AVAILABLE, r):
        if "cx" in combo and "cx2" in combo and len(combo) == 2:
            continue                          # deux versions du même modèle : pas un vrai ensemble
        pv = avg("val", combo).argmax(1)
        cands.append((combo, acc(y_val, pv), mf1(y_val, pv)))
cands.sort(key=lambda c: -c[1])

print(f"Référence (EffNet + ConvNeXt v1, poids {W.round(3)}) : VAL acc={ref_acc:.4f}  F1={ref_f1:.4f}\n")
print(f"{'Ensemble (moyenne simple)':40s} VAL acc  VAL F1")
for combo, a_, f_ in cands:
    print(f"{' + '.join(combo):40s} {a_:.4f}  {f_:.4f}")

eligibles = [c for c in cands if c[2] >= ref_f1 - 0.005]
best_c = eligibles[0] if eligibles else None
if best_c and best_c[1] >= ref_acc + 0.003:
    FINAL_NAMES = list(best_c[0])
    probs_val, probs_test, probs_ph2 = (avg(s, FINAL_NAMES) for s in ("val", "test", "ph2"))
    CHOIX = "Moyenne " + " + ".join(MODEL_LABELS[n] for n in FINAL_NAMES)
else:
    FINAL_NAMES = REF_NAMES
    probs_val, probs_test, probs_ph2 = ref_val, ref_test, ref_ph2
    CHOIX = "Ensemble de référence (aucun candidat ne le bat nettement sur VAL)"
print("\n→ Retenu (décidé sur VAL) :", CHOIX)

Référence (EffNet + ConvNeXt v1, poids [0.488 0.512]) : VAL acc=0.8743  F1=0.8295

Ensemble (moyenne simple)                VAL acc  VAL F1
eff + cx2                                0.8797  0.8335
eff + cx + cx2 + b4                      0.8794  0.8419
eff + cx2 + b4                           0.8789  0.8437
eff + cx + cx2                           0.8775  0.8328
cx + cx2 + b4                            0.8762  0.8333
eff + cx + b4                            0.8759  0.8400
eff + cx                                 0.8735  0.8287
cx + b4                                  0.8729  0.8279
cx2 + b4                                 0.8724  0.8343
eff + b4                                 0.8702  0.8272

→ Retenu (décidé sur VAL) : Moyenne EfficientNetV2-S + CBAM + ConvNeXt-Tiny (v2)


## 2.9 — Calibration sur VAL : température + **un seul facteur mélanome**
Remplace les 7 seuils optimisés (instables : ex. `vasc=0.50` appris sur quelques dizaines d'images).
Le facteur `k` multiplie la probabilité mélanome ; on prend le plus petit `k` qui atteint la sensibilité cible sur VAL.
La température ne change pas les décisions, seulement la confiance affichée.

In [17]:
def temp_scale(probs, T):
    s = np.exp(np.log(np.clip(probs, 1e-12, 1.0)) / T)
    return s / s.sum(axis=1, keepdims=True)

def fit_temperature(pv, y):
    nll = lambda x: log_loss(y, temp_scale(pv, float(x[0])), labels=np.arange(NUM_CLASSES))
    return float(minimize(nll, x0=[1.0], bounds=[(0.05, 5.0)], method="L-BFGS-B").x[0])

def predict_mel_factor(probs, k):
    q = probs.copy(); q[:, MEL] *= k
    return q.argmax(1)

def fit_mel_factor(pv, y, target):
    for k in np.exp(np.linspace(0, np.log(50), 3000)):
        if s_mel(y, predict_mel_factor(pv, k)) >= target:
            return float(k)
    return 50.0

s_mel  = lambda yt, yp: ((yt == MEL) & (yp == MEL)).sum() / max((yt == MEL).sum(), 1)
sp_mel = lambda yt, yp: ((yt != MEL) & (yp != MEL)).sum() / max((yt != MEL).sum(), 1)

CIBLE_SENS_MEL = 0.85
T_final = fit_temperature(probs_val, y_val)
K_MEL   = fit_mel_factor(probs_val, y_val, CIBLE_SENS_MEL)
print(f"T = {T_final:.3f} | facteur mélanome k = {K_MEL:.3f}")
for nom, yp in [("brut", probs_val.argmax(1)), ("calibré", predict_mel_factor(probs_val, K_MEL))]:
    print(f"VAL {nom:8s} Acc={acc(y_val, yp):.4f}  F1={mf1(y_val, yp):.4f}  "
          f"SensMel={s_mel(y_val, yp):.4f}  SpecMel={sp_mel(y_val, yp):.4f}")

T = 0.673 | facteur mélanome k = 2.581
VAL brut     Acc=0.8797  F1=0.8335  SensMel=0.7416  SpecMel=0.9673
VAL calibré  Acc=0.8616  F1=0.8202  SensMel=0.8502  SpecMel=0.9151


## 2.10 — TEST (une seule fois) : IC 95 % par **bootstrap groupé par lésion**, AUC avec IC, gain apparié
Les mêmes tirages servent à toutes les métriques et à la comparaison avec la référence (bootstrap apparié).

In [18]:
def cluster_boot_indices(groups, n=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    uniq, inv = np.unique(groups, return_inverse=True)
    order = np.argsort(inv, kind="stable")
    starts = np.r_[0, np.cumsum(np.bincount(inv))]
    members = [order[starts[g]:starts[g + 1]] for g in range(len(uniq))]
    return [np.concatenate([members[g] for g in rng.integers(0, len(uniq), len(uniq))]) for _ in range(n)]

def macro_auc(yt, probs):
    try:
        return roc_auc_score(label_binarize(yt, classes=np.arange(NUM_CLASSES)), probs, average="macro")
    except ValueError:
        return np.nan

def ci(vals):
    vals = np.asarray(vals, float); vals = vals[np.isfinite(vals)]
    return np.percentile(vals, [2.5, 97.5])

BOOT = cluster_boot_indices(test_df["lesion_id"].values)
yt_raw = probs_test.argmax(1)
yt_cal = predict_mel_factor(probs_test, K_MEL)

def report_test(nom, yp, probs):
    print(f"\n=== {nom} ===")
    out = {}
    for mname, fn in [("Accuracy", acc), ("Macro-F1", mf1), ("Sensibilité mel", s_mel), ("Spécificité mel", sp_mel)]:
        v = fn(y_test, yp); lo, hi = ci([fn(y_test[i], yp[i]) for i in BOOT])
        out[mname] = (float(v), float(lo), float(hi))
        print(f"  {mname:16s} {v:.4f}   IC95% [{lo:.4f} – {hi:.4f}]")
    v = macro_auc(y_test, probs); lo, hi = ci([macro_auc(y_test[i], probs[i]) for i in BOOT[:500]])
    out["Macro AUC"] = (float(v), float(lo), float(hi))
    print(f"  {'Macro AUC':16s} {v:.4f}   IC95% [{lo:.4f} – {hi:.4f}]")
    return out

res_final_raw = report_test(f"TEST brut — {CHOIX}", yt_raw, probs_test)
res_final_cal = report_test(f"TEST calibré (sens. mel ≥ {CIBLE_SENS_MEL} sur VAL)", yt_cal, probs_test)

print("\nGain vs ensemble de référence (bootstrap apparié, groupé par lésion) :")
ref_raw_t = ref_test.argmax(1)
gains_final = {}
for mname, fn in [("Accuracy", acc), ("Macro-F1", mf1)]:
    d = fn(y_test, yt_raw) - fn(y_test, ref_raw_t)
    lo, hi = ci([fn(y_test[i], yt_raw[i]) - fn(y_test[i], ref_raw_t[i]) for i in BOOT])
    gains_final[mname] = (float(d), float(lo), float(hi))
    verdict = "✅ significatif" if lo > 0 else ("❌ moins bon" if hi < 0 else "≈ non significatif")
    print(f"  {mname:9s} {d:+.4f}   IC95% [{lo:+.4f} ; {hi:+.4f}]   {verdict}")

print("\n--- Rapport par classe (brut) ---")
print(classification_report(y_test, yt_raw, labels=np.arange(NUM_CLASSES),
                            target_names=CLASS_NAMES_FULL, digits=4, zero_division=0))
print("Matrice de confusion (brut ; lignes = vrai) :")
print(confusion_matrix(y_test, yt_raw, labels=np.arange(NUM_CLASSES)))

print("\nCompromis sensibilité / spécificité (facteur choisi sur VAL, appliqué au TEST) :")
print("Cible VAL | Acc test | Macro-F1 | Sens. mel | Spéc. mel")
for cible in [0.70, 0.75, 0.80, 0.85, 0.90]:
    yp = predict_mel_factor(probs_test, fit_mel_factor(probs_val, y_val, cible))
    print(f"   {cible:.2f}   |  {acc(y_test, yp):.4f}  |  {mf1(y_test, yp):.4f}  |  "
          f"{s_mel(y_test, yp):.4f}   |  {sp_mel(y_test, yp):.4f}")


=== TEST brut — Moyenne EfficientNetV2-S + CBAM + ConvNeXt-Tiny (v2) ===
  Accuracy         0.8737   IC95% [0.8611 – 0.8863]
  Macro-F1         0.8311   IC95% [0.8015 – 0.8562]
  Sensibilité mel  0.7493   IC95% [0.7110 – 0.7876]
  Spécificité mel  0.9639   IC95% [0.9562 – 0.9714]
  Macro AUC        0.9813   IC95% [0.9771 – 0.9848]

=== TEST calibré (sens. mel ≥ 0.85 sur VAL) ===
  Accuracy         0.8518   IC95% [0.8383 – 0.8641]
  Macro-F1         0.8171   IC95% [0.7871 – 0.8431]
  Sensibilité mel  0.8449   IC95% [0.8130 – 0.8741]
  Spécificité mel  0.9081   IC95% [0.8962 – 0.9196]
  Macro AUC        0.9813   IC95% [0.9771 – 0.9848]

Gain vs ensemble de référence (bootstrap apparié, groupé par lésion) :
  Accuracy  +0.0029   IC95% [-0.0013 ; +0.0073]   ≈ non significatif
  Macro-F1  +0.0002   IC95% [-0.0101 ; +0.0099]   ≈ non significatif

--- Rapport par classe (brut) ---
                               precision    recall  f1-score   support

            Actinic keratoses     0.7600

## 2.11 — PH2 (validation externe) : même point de fonctionnement, AUC avec IC, et AUC par modèle

In [19]:
yb_ph2 = (y_ph2 == MEL).astype(int)
rng = np.random.default_rng(SEED)
BOOT_PH2 = [rng.integers(0, len(yb_ph2), len(yb_ph2)) for _ in range(2000)]   # PH2 : 1 image = 1 lésion

def auc_mel(y, pm):
    return roc_auc_score(y, pm) if 0 < y.sum() < len(y) else np.nan

sens_b = lambda a, b: ((a == 1) & (b == 1)).sum() / max((a == 1).sum(), 1)
spec_b = lambda a, b: ((a == 0) & (b == 0)).sum() / max((a == 0).sum(), 1)

res_ph2_final = {}
for nom, yp in [("brut", (probs_ph2.argmax(1) == MEL).astype(int)),
                ("calibré", (predict_mel_factor(probs_ph2, K_MEL) == MEL).astype(int))]:
    print(f"\n=== PH2 {nom} (197 images, mel vs non-mel) ===")
    res_ph2_final[nom] = {}
    for mname, fn in [("Accuracy", acc), ("Sensibilité mel", sens_b), ("Spécificité mel", spec_b)]:
        v = fn(yb_ph2, yp); lo, hi = ci([fn(yb_ph2[i], yp[i]) for i in BOOT_PH2])
        res_ph2_final[nom][mname] = (float(v), float(lo), float(hi))
        print(f"  {mname:16s} {v:.4f}   IC95% [{lo:.4f} – {hi:.4f}]")

v = auc_mel(yb_ph2, probs_ph2[:, MEL]); lo, hi = ci([auc_mel(yb_ph2[i], probs_ph2[i, MEL]) for i in BOOT_PH2])
res_ph2_final["AUC mel"] = (float(v), float(lo), float(hi))
print(f"\nAUC mélanome PH2 (retenu) : {v:.4f}   IC95% [{lo:.4f} – {hi:.4f}]")

print("\nAUC mélanome PH2 par modèle (lequel généralise le mieux hors ISIC ?) :")
for n in AVAILABLE:
    print(f"  {MODEL_LABELS[n]:26s} {auc_mel(yb_ph2, P['ph2'][n][:, MEL]):.4f}")


=== PH2 brut (197 images, mel vs non-mel) ===
  Accuracy         0.8426   IC95% [0.7919 – 0.8934]
  Sensibilité mel  0.5385   IC95% [0.3958 – 0.6667]
  Spécificité mel  0.9517   IC95% [0.9130 – 0.9857]

=== PH2 calibré (197 images, mel vs non-mel) ===
  Accuracy         0.8579   IC95% [0.8071 – 0.9086]
  Sensibilité mel  0.7115   IC95% [0.5854 – 0.8277]
  Spécificité mel  0.9103   IC95% [0.8609 – 0.9565]

AUC mélanome PH2 (retenu) : 0.8496   IC95% [0.7793 – 0.9138]

AUC mélanome PH2 par modèle (lequel généralise le mieux hors ISIC ?) :
  EfficientNetV2-S + CBAM    0.8590
  ConvNeXt-Tiny (v1)         0.8146
  ConvNeXt-Tiny (v2)         0.7999
  EfficientNet-B4            0.8393


## 2.12 — Sauvegarde et bilan

In [20]:
np.save(f"{OUT_DIR}/final_probs_val.npy",  probs_val)
np.save(f"{OUT_DIR}/final_probs_test.npy", probs_test)
np.save(f"{OUT_DIR}/final_probs_ph2.npy",  probs_ph2)

pd.DataFrame({"image_uid": test_df["image_uid"].values, "lesion_id": test_df["lesion_id"].values,
              "true": [IDX_TO_CLASS[i] for i in y_test],
              "pred_brut": [IDX_TO_CLASS[i] for i in yt_raw],
              "pred_calibre": [IDX_TO_CLASS[i] for i in yt_cal],
              "confiance": temp_scale(probs_test, T_final).max(1)}
             ).to_csv(f"{OUT_DIR}/final_predictions_test.csv", index=False)

final_config = {
    "choix": CHOIX, "modeles": FINAL_NAMES, "combinaison": "moyenne simple des probabilités TTA",
    "checkpoints": {"cx2": CX_NEW if os.path.exists(CX_NEW) else None,
                    "b4": B4_FINAL if os.path.exists(B4_FINAL) else None},
    "entrainement": {"convnext_v2_val_f1": float(best_cx2), "convnext_v2_epoques": int(ep_cx2),
                     "b4_val_f1": float(best_b4) if "best_b4" in globals() else None},
    "temperature": T_final, "facteur_mel": K_MEL, "cible_sens_mel": CIBLE_SENS_MEL,
    "test_brut": res_final_raw, "test_calibre": res_final_cal, "gain_vs_reference": gains_final,
    "ph2": res_ph2_final, "duree_h": round((time.time() - T0) / 3600, 2),
}
with open(f"{OUT_DIR}/run_config_final.json", "w") as f:
    json.dump(final_config, f, indent=2, ensure_ascii=False)

a, lo, hi = res_final_raw["Accuracy"]
print("═" * 70)
print(f"MODÈLE RETENU : {CHOIX}")
print(f"TEST accuracy brute : {a:.4f}  IC95% [{lo:.4f} – {hi:.4f}]")
for cible in (0.88, 0.89, 0.90):
    etat = "✅ atteint" if a >= cible else "❌ non atteint"
    solide = " (borne basse de l'IC au-dessus)" if lo >= cible else ""
    print(f"  Objectif {cible:.0%} : {etat}{solide}")
a, lo, hi = res_final_cal["Accuracy"]
print(f"TEST accuracy calibrée (sens. mel ≥ {CIBLE_SENS_MEL}) : {a:.4f}  IC95% [{lo:.4f} – {hi:.4f}]")
print(f"Durée partie 2 : {(time.time() - T0) / 3600:.2f} h")
print("═" * 70)

══════════════════════════════════════════════════════════════════════
MODÈLE RETENU : Moyenne EfficientNetV2-S + CBAM + ConvNeXt-Tiny (v2)
TEST accuracy brute : 0.8737  IC95% [0.8611 – 0.8863]
  Objectif 88% : ❌ non atteint
  Objectif 89% : ❌ non atteint
  Objectif 90% : ❌ non atteint
TEST accuracy calibrée (sens. mel ≥ 0.85) : 0.8518  IC95% [0.8383 – 0.8641]
Durée partie 2 : 5.69 h
══════════════════════════════════════════════════════════════════════
